# 算子组合与 Softmax

这一节开始练习“把公式翻译成 kernel”。你会先实现几类常见激活函数，再实现稳定 Softmax。它们看起来都不长，但能覆盖中高级算子里非常常见的组合方式：elementwise、reduction、广播、动态 batch 和 loop 分块。

学习重点不是记住每个 API 的名字，而是看清一个公式如何拆成 PyPTO 可以记录的计算图。只要这一步熟练，后面写 LayerNorm、FFN 和 Attention 时就不会被更长的代码吓住。

## 1. 学完后你应该能够

学完这一节后，可以回到这里检查自己是否已经做到：

1. 解释 SiLU、GELU、SwiGLU、GeGLU 的公式和输入输出关系。
2. 看懂 PyPTO kernel 中临时变量如何对应计算图节点。
3. 理解 `configure_tiling` 为什么只影响执行组织，不改变数学含义。
4. 说明 Softmax 为什么要先减 `row_max`。
5. 理解 `pypto.Tensor([pypto.DYNAMIC, ...], dtype)`、`pypto.loop` 和 batch 分块之间的关系。
6. 能用 PyTorch reference 写出每个算子的验证闭环。

## 2. 这一节会依次练什么

| 练习内容 | 输入形式 | 主要关注点 |
| --- | --- | --- |
| SiLU | 单个 Tensor | 单输入 elementwise 组合，`x * sigmoid(x)` |
| GELU 近似 | 单个 Tensor | 常数乘法、近似公式、近似误差 |
| SwiGLU | `gate` 和 `up` 两个 Tensor | 双输入门控激活，两路输入逐元素对齐 |
| GeGLU | `gate` 和 `up` 两个 Tensor | GELU 门控分支，近似公式复用 |
| Softmax | 四维 Tensor | 稳定 Softmax、动态 batch、loop 分块 |

前四个练习都属于激活函数组合，核心是 elementwise operation 串联；最后的 Softmax 会引入 reduction、广播和动态 shape。阅读时先确认公式，再确认输入输出 shape，最后看 kernel 是否按同一个公式写回输出。

### 2.1 建议运行顺序

这一节的代码按“先工具、再 kernel、再测试”的顺序排列。运行时建议保持这个顺序：

| 步骤 | 你会看到什么 | 为什么先看它 |
| --- | --- | --- |
| 环境与工具 | `get_device()`、`RUN_MODE`、`configure_tiling()`、reference 函数 | 先确定真实 Tensor 在哪里、kernel 用什么模式运行。 |
| 单输入激活函数 | SiLU、GELU 近似 kernel | 先用最短公式熟悉 elementwise 组合。 |
| 双输入门控激活 | SwiGLU、GeGLU kernel | 再观察两路输入如何逐元素对齐。 |
| Softmax | `amax`、`exp`、`sum`、`loop` | 最后把 reduction、广播和动态 batch 串起来。 |
| 验证函数 | `test_*()` 与 PyTorch reference | 对照 shape、公式和误差阈值，确认 kernel 写对。 |

如果正在 Notebook 中跟着运行，先执行环境准备单元，再执行对应 kernel 定义单元，最后运行测试单元。修改过 JIT kernel 后，建议重新运行环境准备单元和当前 kernel 定义单元，避免旧的记录状态影响结果。

## 3. 环境准备



In [ ]:
import os
os.environ['TILE_FWK_DEVICE_ID'] = '0'
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'
import torch
import pypto
import numpy as np
from numpy.testing import assert_allclose
import torch_npu


def get_device():
    device_id = int(os.environ.get("TILE_FWK_DEVICE_ID", "0"))
    return f"npu:{device_id}"


device = get_device()
RUN_MODE = pypto.RunMode.NPU

def configure_tiling(x):
    if len(x.shape) >= 2:
        tile_list = [32 for _ in range(len(x.shape))]
        pypto.set_vec_tile_shapes(*tile_list)
    else:
        pypto.set_vec_tile_shapes(32, 128)


def silu_golden(x: torch.Tensor) -> torch.Tensor:
    return x * torch.sigmoid(x)


def gelu_golden(x: torch.Tensor) -> torch.Tensor:
    return torch.nn.functional.gelu(x)


def gelu_approx_golden(x: torch.Tensor) -> torch.Tensor:
    return x * torch.sigmoid(1.702 * x)


def swiglu_golden(gate: torch.Tensor, up: torch.Tensor) -> torch.Tensor:
    return (gate * torch.sigmoid(gate)) * up


def geglu_golden(gate: torch.Tensor, up: torch.Tensor) -> torch.Tensor:
    return torch.nn.functional.gelu(gate) * up


print("TILE_FWK_DEVICE_ID:", os.environ.get("TILE_FWK_DEVICE_ID", "<not set>"))
print("device:", device)
print("run_mode:", RUN_MODE)
print("pypto:", pypto.__file__)


### 3.1 通用工具逐段说明

| 函数或变量 | 作用 | 阅读时关注点 |
| --- | --- | --- |
| `get_device()` | 选择 NPU | `torch.Tensor` 的真实 device 由这里决定。 |
| `RUN_MODE` | 传给 `@pypto.frontend.jit`，决定 kernel 运行模式 | NPU 环境为 `NPU`，无 NPU 时为 `SIM`。 |
| `configure_tiling(x)` | 根据输入维度设置 vec tile，服务 elementwise 激活函数 | 只影响执行组织，不改变公式。 |
| `*_golden` | PyTorch reference，用来验证 PyPTO kernel 输出 | reference 的公式和 dtype 要与 kernel 对齐。 |

这里最重要的分工是：`torch.Tensor` 承载真实数据，`pypto.Tensor(...)` 描述 kernel 参数。调用 kernel 时，真实的 `torch.Tensor` 会传入这些参数位置；在 JIT 函数内部，`x * pypto.sigmoid(x)` 这类表达式会被记录成 PyPTO 计算图，而不是立即按普通 Python 标量逻辑求值。

`configure_tiling(x)` 的规则很简单：二维或更高维输入就按每个维度给出 tile；一维输入则给一个默认二元 tile。Tile 可以理解为设备侧处理数据的小块形状，决定怎样组织向量计算和数据搬运，但不会改变 `x * sigmoid(x)`、`exp / sum(exp)` 这类数学结果。

## 4. 单输入激活函数：SiLU 与 GELU

先看最简单的一类：输入只有一个 Tensor，输出 shape 与输入相同。

| 激活函数 | 公式 | 输入 shape | 输出 shape |
| --- | --- | --- | --- |
| SiLU | `x * sigmoid(x)` | `[32, 128]` | `[32, 128]` |
| GELU 近似 | `x * sigmoid(1.702 * x)` | `[32, 128]` | `[32, 128]` |

这两个算子都是 elementwise：每个位置独立计算，不需要跨行、跨列归约。尤其注意这里的gelu是近似实现。

In [ ]:
@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def silu_activation_kernel(
    x: pypto.Tensor(),
    out: pypto.Tensor()):
    configure_tiling(x)
    out[:] = x * pypto.sigmoid(x)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def gelu_activation_kernel(
    x: pypto.Tensor(),
    out: pypto.Tensor()):
    configure_tiling(x)
    x_scaled = x * 1.702
    out[:] = x * pypto.sigmoid(x_scaled)


### 4.1 Kernel 代码逐行理解

SiLU 的 kernel 很短，但包含了 PyPTO elementwise 算子的完整结构：

| 代码 | 含义 |
| --- | --- |
| `@pypto.frontend.jit(...)` | 这个函数会进入 PyPTO 编译流程，不是普通 Python 逐行解释执行。 |
| `x: pypto.Tensor()` | `x` 是输入 Tensor 描述，真实数据由调用时的 `torch.Tensor` 提供。 |
| `out: pypto.Tensor()` | `out` 是输出 Tensor 描述，由 host 侧提前分配。 |
| `configure_tiling(x)` | 按输入维度设置 vec tile，给后端一个分块执行提示。 |
| `pypto.sigmoid(x)` | 记录一个 sigmoid 节点，shape 与 `x` 相同。 |
| `x * pypto.sigmoid(x)` | 记录逐元素乘法节点，得到 SiLU 结果。 |
| `out[:] = ...` | 把最终结果写回输出 Tensor 的全部位置。 |

GELU 近似只比 SiLU 多一步 `x_scaled = x * 1.702`。这一步仍然是 elementwise，不改变 shape：

```text
x:        [32, 128], BF16
x_scaled: [32, 128], BF16
out:      [32, 128], BF16
```

In [ ]:
def test_silu() -> None:
    device_local = get_device()
    shape = (32, 128)
    x_torch = torch.randn(shape, dtype=torch.bfloat16, device=device_local)
    out_torch = torch.empty(shape, dtype=torch.bfloat16, device=device_local)

    silu_activation_kernel(x_torch, out_torch)
    expected = silu_golden(x_torch)
    max_diff = (out_torch - expected).abs().max().item()

    print(f"SiLU input shape: {x_torch.shape}")
    print(f"SiLU output shape: {out_torch.shape}")
    print(f"SiLU max difference: {max_diff:.6f}")
    assert max_diff < 1e-1, "SiLU result mismatch!"


def test_gelu() -> None:
    device_local = get_device()
    shape = (32, 128)
    x_torch = torch.randn(shape, dtype=torch.bfloat16, device=device_local)
    out_torch = torch.empty(shape, dtype=torch.bfloat16, device=device_local)

    gelu_activation_kernel(x_torch, out_torch)
    expected_full = gelu_golden(x_torch)
    expected_approx = gelu_approx_golden(x_torch)
    max_diff_full = (out_torch - expected_full).abs().max().item()
    max_diff_approx = (out_torch - expected_approx).abs().max().item()

    print(f"GELU input shape: {x_torch.shape}")
    print(f"GELU output shape: {out_torch.shape}")
    print(f"GELU max difference vs torch.gelu: {max_diff_full:.6f}")
    print(f"GELU max difference vs approximation: {max_diff_approx:.6f}")
    assert max_diff_full < 1e-1, "GELU result mismatch!"


test_silu()
test_gelu()


### 4.2 验证代码怎么读

每个测试函数都遵循同一套验证闭环：

1. 用 `torch.randn` 在 host 侧构造真实输入。
2. 用 `torch.empty` 或 `torch.empty_like` 提前分配输出。
3. 调用 PyPTO kernel，把结果写入输出 Tensor。
4. 用 PyTorch 写 reference。
5. 比较最大误差，并在 NPU 模式下断言误差阈值。

逐段看 `test_silu()`：

| 代码 | 作用 |
| --- | --- |
| `shape = (32, 128)` | 固定一个二维输入，便于观察 elementwise shape 不变。 |
| `torch.randn(..., dtype=torch.bfloat16, device=device_local)` | 在目标设备上生成 BF16 输入，用来观察低精度激活函数的误差。 |
| `torch.empty(shape, ...)` | 只分配输出内存，不初始化内容，后续由 kernel 写满。 |
| `silu_activation_kernel(x_torch, out_torch)` | 触发 PyPTO kernel 执行。 |
| `expected = silu_golden(x_torch)` | 用 PyTorch 写出同样公式。 |
| `max_diff = ...abs().max().item()` | 汇总最大绝对误差，方便定位精度问题。 |

GELU 这里同时打印两个误差：一个对比 PyTorch 完整 GELU，一个对比 `x * sigmoid(1.702 * x)` 近似公式。PyPTO kernel 实现的是近似公式，因此和近似 reference 应该更接近；和完整 GELU 的误差来自公式近似，而不是 PyPTO 写法本身。BF16 场景下误差会比 FP32 更明显，这里使用 `1e-1` 作为较宽松的判断阈值。

**预期输出说明**

运行成功后，应看到 SiLU 和 GELU 的输入、输出 shape 都是 `torch.Size([32, 128])`。SiLU 的最大误差应很小；GELU 对比近似公式的误差应比对比完整 `torch.nn.functional.gelu` 更小。若在非 NPU 环境中运行，Notebook 主要用于阅读代码结构，具体执行能力取决于本地 PyPTO SIM 支持情况。

## 5. 双输入门控激活：SwiGLU 与 GeGLU

门控激活函数有两路输入：`gate` 和 `up`。可以把它理解成：`gate` 分支先计算一个非线性权重，`up` 分支提供被调制的内容。

| 激活函数 | gate 分支 | 输出公式 |
| --- | --- | --- |
| SwiGLU | `gate * sigmoid(gate)` | `(gate * sigmoid(gate)) * up` |
| GeGLU | `gate * sigmoid(1.702 * gate)` | `GELU(gate) * up` |

这类结构会在 FFN 中再次出现，因为很多大模型的 FFN 都使用门控分支。

In [ ]:
@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def swiglu_activation_kernel(
    gate: pypto.Tensor(),
    up: pypto.Tensor(),
    out: pypto.Tensor()):
    configure_tiling(gate)
    sigmoid = pypto.sigmoid(gate)
    swish = gate * sigmoid
    out[:] = swish * up


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def geglu_activation_kernel(
    gate: pypto.Tensor(),
    up: pypto.Tensor(),
    out: pypto.Tensor()):
    configure_tiling(gate)
    gate_scaled = gate * 1.702
    gelu_gate = gate * pypto.sigmoid(gate_scaled)
    out[:] = gelu_gate * up


### 5.1 门控 kernel 逐段说明

SwiGLU 可以拆成三步：

```text
sigmoid = sigmoid(gate)
swish = gate * sigmoid
out = swish * up
```

逐行对应关系如下：

| 代码 | shape 变化 | 含义 |
| --- | --- | --- |
| `configure_tiling(gate)` | 不改变 shape | 用 `gate` 的维度设置 elementwise tile。 |
| `sigmoid = pypto.sigmoid(gate)` | `[32, 128] -> [32, 128]` | 计算 gate 分支的 sigmoid 权重。 |
| `swish = gate * sigmoid` | `[32, 128] -> [32, 128]` | 得到 SiLU/Swish 形式的 gate 分支。 |
| `out[:] = swish * up` | `[32, 128] -> [32, 128]` | 用 gate 分支调制 `up` 分支。 |

GeGLU 也可以拆成三步：

```text
gate_scaled = gate * 1.702
gelu_gate = gate * sigmoid(gate_scaled)
out = gelu_gate * up
```

`gate_scaled` 和 `gelu_gate` 都是临时计算图节点。它们的存在是为了把复杂公式拆成可读的小表达式；后端仍然可以根据能力进行融合或调度优化。二者的输入、输出 shape 都是 `[32, 128]`。`gate` 和 `up` 必须能逐元素对齐；如果 shape 不一致，就需要符合广播规则，否则无法完成逐元素乘法。

In [ ]:
def test_swiglu() -> None:
    device_local = get_device()
    shape = (32, 128)
    gate_torch = torch.randn(shape, dtype=torch.bfloat16, device=device_local)
    up_torch = torch.randn(shape, dtype=torch.bfloat16, device=device_local)
    out_torch = torch.empty(shape, dtype=torch.bfloat16, device=device_local)

    swiglu_activation_kernel(gate_torch, up_torch, out_torch)
    expected = swiglu_golden(gate_torch, up_torch)
    max_diff = (out_torch - expected).abs().max().item()

    print(f"SwiGLU gate shape: {gate_torch.shape}")
    print(f"SwiGLU up shape: {up_torch.shape}")
    print(f"SwiGLU output shape: {out_torch.shape}")
    print(f"SwiGLU max difference: {max_diff:.6f}")
    assert max_diff < 1e-1, "SwiGLU result mismatch!"


def test_geglu() -> None:
    device = get_device()
    shape = (32, 128)
    gate_torch = torch.randn(shape, dtype=torch.bfloat16, device=device)
    up_torch = torch.randn(shape, dtype=torch.bfloat16, device=device)
    out_torch = torch.empty(shape, dtype=torch.bfloat16, device=device)
    # Execute
    geglu_activation_kernel(gate_torch, up_torch, out_torch)

    # Verify
    expected = geglu_golden(gate_torch, up_torch)
    max_diff = (out_torch - expected).abs().max().item()

    print(f"Gate shape: {gate_torch.shape}")
    print(f"Up shape: {up_torch.shape}")
    print(f"Output shape: {out_torch.shape}")
    print(f"Max difference: {max_diff:.6f}")
    assert max_diff < 1e-1, "Result mismatch!"


test_swiglu()
test_geglu()


### 5.2 双输入验证逻辑

双输入算子验证时要额外检查两路输入 shape：

```text
gate: [32, 128]
up:   [32, 128]
out:  [32, 128]
```

只要 `gate` 和 `up` 在每个位置能一一对应，门控乘法就不会改变 shape。后续 FFN 中的 SwiGLU 会把这里的 `gate` 和 `up` 换成两个 matmul 投影结果，公式结构保持不变。

验证 GeGLU 时也要区分“完整 GELU reference”和“近似 GELU reference”。本节的 kernel 使用 `gate * sigmoid(1.702 * gate)`，因此：

```python
expected_approx = gelu_approx_golden(gate_torch) * up_torch
```

才是完全同公式的 reference；`torch.nn.functional.gelu(gate_torch) * up_torch` 用于观察近似误差是否在可接受范围内。

### 5.3 门控激活与 FFN 的关系

单独看 SwiGLU/GeGLU，它们只是两个输入逐元素相乘；放到 FFN 里，`gate` 和 `up` 通常来自两组线性投影：

```text
gate = x @ w_gate
up   = x @ w_up
hidden = activation(gate) * up
out = hidden @ w_down
```

所以本节不是孤立地学习激活函数，而是在为 4.3 的 FFN 做铺垫。只要先把 `activation(gate) * up` 这一步理解清楚，后面看到 matmul 版本时就只需要额外关注投影前后的 shape。

## 6. Softmax：从公式到动态 batch kernel

Softmax 的稳定公式是：

```text
softmax(x) = exp(x - max(x)) / sum(exp(x - max(x)))
```

如果直接计算 `exp(x)`，当 `x` 很大时容易溢出。先减去每一行的最大值，可以把最大元素平移到 0，其它元素小于等于 0，从而让 `exp` 更稳定。

本节 Softmax 输入 shape 是 `[batch, seqlen, head, dim] = [32, 32, 1, 256]`，沿最后一维 `dim=-1` 做归一化。

In [ ]:
def softmax_core(x: pypto.Tensor) -> pypto.Tensor:
    row_max = pypto.amax(x, dim=-1, keepdim=True)
    sub = x - row_max
    exp = pypto.exp(sub)
    esum = pypto.sum(exp, dim=-1, keepdim=True)
    return exp / esum


### 6.1 `softmax_core` 逐行理解

| 代码 | 输入 shape | 输出 shape | 含义 |
| --- | --- | --- | --- |
| `row_max = pypto.amax(x, dim=-1, keepdim=True)` | `[B, S, H, D]` | `[B, S, H, 1]` | 每个 batch、token、head 内，沿最后一维取最大值。 |
| `sub = x - row_max` | `[B, S, H, D]` 和 `[B, S, H, 1]` | `[B, S, H, D]` | `row_max` 通过广播减到每个元素上。 |
| `exp = pypto.exp(sub)` | `[B, S, H, D]` | `[B, S, H, D]` | 对平移后的分数取指数。 |
| `esum = pypto.sum(exp, dim=-1, keepdim=True)` | `[B, S, H, D]` | `[B, S, H, 1]` | 计算归一化分母。 |
| `return exp / esum` | `[B, S, H, D]` 和 `[B, S, H, 1]` | `[B, S, H, D]` | 广播除法，得到概率分布。 |

用一个小向量直观看：如果最后一维是 `[2.0, 4.0, 1.0]`，先减最大值 4.0，得到 `[-2.0, 0.0, -3.0]`。最大指数项变成 `exp(0)=1`，其它指数项小于 1，溢出风险显著降低。因为 Softmax 对同一行加减常数不改变结果，所以这种平移是数学等价的。

`keepdim=True` 很关键。如果去掉它，`row_max` 和 `esum` 的最后一维会消失，后续广播就不如 `[B, S, H, 1]` 这样直观。这里保留维度，是为了让 shape 变化一眼可见。

In [ ]:
@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def softmax_kernel(
    input_tensor: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32),
    output_tensor: pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32)):
    bs, seqlen, head, dim = input_tensor.shape
    tile_b = 1
    b_loop = bs // tile_b

    pypto.set_vec_tile_shapes(1, 4, 1, 64)

    for idx in pypto.loop(0, b_loop, 1, name="LOOP_L0_bIdx", idx_name="idx"):
        b_offset = idx * tile_b
        b_offset_end = (idx + 1) * tile_b
        input_view = input_tensor[b_offset:b_offset_end, :seqlen, :head, :dim]
        softmax_out = softmax_core(input_view)
        output_tensor[b_offset:b_offset_end, ...] = softmax_out


### 6.2 `softmax_kernel` 逐段理解

这个 kernel 比前面的激活函数多了动态 batch 和 loop：

| 代码 | 含义 |
| --- | --- |
| `pypto.Tensor([pypto.DYNAMIC, ...], pypto.DT_FP32)` | 第 0 维 batch 是动态的，其它维度从运行时输入继承。 |
| `bs, seqlen, head, dim = input_tensor.shape` | 从输入 Tensor 描述中取出四个维度。 |
| `tile_b = 1` | 每次处理 1 个 batch tile。 |
| `b_loop = bs // tile_b` | 计算需要循环多少个 batch tile。当前 shape 下是 32 次。 |
| `pypto.set_vec_tile_shapes(1, 4, 1, 64)` | 按 `[batch_tile, seq_tile, head_tile, dim_tile]` 设置向量 tile。 |
| `pypto.loop(...)` | 在 PyPTO 计算图中描述循环，不是普通 Python `range`。 |
| `b_offset = idx * tile_b` | 计算当前 tile 的 batch 起点。 |
| `b_offset_end = (idx + 1) * tile_b` | 计算当前 tile 的 batch 终点。 |
| `input_view = input_tensor[...]` | 取当前 batch tile 的局部输入，shape 为 `[1, S, H, D]`。 |
| `softmax_core(input_view)` | 对局部 tile 沿最后一维做 Softmax。 |
| `output_tensor[b_offset:b_offset_end, ...] = softmax_out` | 把当前 tile 结果写回输出对应位置。 |

以本节输入 `[32, 32, 1, 256]` 为例，前三次循环可以这样理解：

```text
idx = 0 -> 处理 batch [0:1]
idx = 1 -> 处理 batch [1:2]
idx = 2 -> 处理 batch [2:3]
...
idx = 31 -> 处理 batch [31:32]
```

每次循环内部的 Softmax 都只在最后一维 `dim=256` 上归一化，batch 分块不会改变数学含义。这个例子只把 batch 维设为动态，因为 `seqlen / head / dim` 在这个场景中相对固定。只动态化必要维度，通常比把所有维度都设成动态更容易优化，也更容易讲清楚 shape。

In [ ]:
def test_softmax() -> None:
    device_local = get_device()
    shape = (32, 32, 1, 256)
    x = torch.rand(shape, dtype=torch.float32, device=device_local)
    y = torch.zeros(shape, dtype=torch.float32, device=device_local)

    softmax_kernel(x, y)
    golden = torch.softmax(x, dim=-1).cpu()
    y_cpu = y.cpu()
    max_diff = np.abs(y_cpu.detach().cpu().numpy() - golden.detach().cpu().numpy()).max()

    print(f"Softmax input shape: {x.shape}")
    print(f"Softmax output shape: {y.shape}")
    print(f"Softmax max difference: {max_diff:.6f}")
    print("First row sum sample:", y_cpu[0, 0, 0, :].sum().item())
    assert_allclose(y_cpu.detach().cpu().numpy(), golden.detach().cpu().numpy(), rtol=3e-3, atol=3e-3)

test_softmax()



### 6.3 Softmax 验证逻辑

Softmax 的验证除了比较 PyTorch reference，还可以观察每一行的和。因为 Softmax 沿最后一维归一化，所以 `y[0, 0, 0, :].sum()` 应该接近 1。

逐段看 `test_softmax()`：

| 代码 | 作用 |
| --- | --- |
| `shape = (32, 32, 1, 256)` | 构造 `[batch, seqlen, head, dim]` 四维输入。 |
| `x = torch.rand(..., dtype=torch.float32)` | Softmax 使用 FP32，更适合观察概率归一化。 |
| `y = torch.zeros(...)` | 提前分配输出，初值会被 kernel 覆盖。 |
| `softmax_kernel(x, y)` | 调用动态 batch 版本的 PyPTO kernel。 |
| `golden = torch.softmax(x, dim=-1)` | PyTorch reference 必须沿最后一维计算。 |
| `np.abs(...).max()` | 比较整个输出 Tensor 的最大误差。 |
| `y_cpu[0, 0, 0, :].sum()` | 抽样检查一行概率和是否接近 1。 |

验证时要注意三点：

1. PyPTO 输出 `y` 和 PyTorch reference `golden` 的 shape 都是 `[32, 32, 1, 256]`。
2. `torch.softmax(x, dim=-1)` 必须和 PyPTO 中 `dim=-1` 对齐。
3. FP32 Softmax 的误差阈值比 BF16 激活函数更严格，这里使用 `rtol=3e-3, atol=3e-3`。

### 6.4 Softmax 调试检查表

如果 Softmax 结果不对，优先按下面顺序检查：

| 检查项 | 常见现象 | 处理方式 |
| --- | --- | --- |
| `dim` 是否一致 | PyPTO 和 PyTorch reference 结果差异很大 | 确认两边都沿 `dim=-1`。 |
| `keepdim` 是否保留 | 广播失败或 shape 不符合预期 | `amax` 和 `sum` 都使用 `keepdim=True`。 |
| 输出切片是否正确 | 只有部分 batch 正确，或后面 batch 被覆盖 | 写回时使用 `output_tensor[b_offset:b_offset_end, ...]`。 |
| dtype 是否一致 | 误差明显变大 | 本例使用 FP32，便于观察 Softmax 的概率归一化。 |
| `tile_b` 是否整除 `bs` | 最后几个 batch 未处理 | 当前例子 `32 // 1` 没有余数；扩展时需要处理尾块。 |

这个检查表也是后续 Attention 里排查 Softmax 子步骤的常用顺序。

## 7. API 速览

| API | 类型 | 作用 |
| --- | --- | --- |
| `pypto.sigmoid` | elementwise | 构造 SiLU、GELU 近似、SwiGLU、GeGLU。 |
| `pypto.exp` | elementwise | 计算 Softmax 的指数项。 |
| `pypto.amax` | reduction | 沿最后一维取最大值，用于数值稳定。 |
| `pypto.sum` | reduction | 沿最后一维求和，得到归一化分母。 |
| `pypto.maximum` | elementwise compare | 后续 ReLU 类写法会频繁使用，本节在概念上与门控类似。 |
| `pypto.loop` | control flow | 按 batch tile 组织 Softmax 输入。 |
| `pypto.set_vec_tile_shapes` | tiling | 指定向量类计算的 tile 组织方式。 |
| `pypto.Tensor([pypto.DYNAMIC, ...], dtype)` | 参数描述 | 声明动态 batch 维。 |

这里最重要的能力是把公式翻译成 PyPTO 计算图，并用 PyTorch reference 验证。

## 8. 常见易混点

| 易混点 | 正确理解 |
| --- | --- |
| `x_scaled = x * 1.702` 会不会创建真实 Tensor 数据 | 在 kernel 中它描述一个计算图节点，真实数据运行时才参与计算。 |
| `1.702 * x` 和 `x * 1.702` 是否完全等价 | 数学等价，但部分 PyPTO 版本对左侧 Python float 的重载不完整，建议 Tensor 放左边。 |
| `configure_tiling` 是否改变公式 | 不改变，只影响设备侧分块执行。 |
| `out[:] = ...` 是否等于普通 Python 赋值 | 在 JIT kernel 中，它表示把计算结果写回输出 Tensor。 |
| GELU PyPTO 输出为何不一定和 `torch.nn.functional.gelu` 完全一致 | 这里实现的是 sigmoid 近似 GELU，不是完整 GELU 公式。 |
| GeGLU 的误差为什么可能比 GELU 更敏感 | GELU 近似误差会再乘以 `up`，`up` 的数值范围会放大或缩小最终误差。 |
| Softmax 为什么使用 `keepdim=True` | 保留最后一维为 1，方便后续和原始输入广播。 |
| 为什么 Softmax 只动态 batch | 这里关注 batch 变化，其它维度保持具体更利于阅读和优化。 |
| `pypto.loop` 是否等同于 Python `range` | 它在 kernel 计算图里描述循环，适合表达设备侧分块计算。 |

## 9. 课后练习

本节练习用于复盘激活函数组合、稳定 Softmax 和动态 batch 分块写回。题型包含选择题和填空题。

1. （填空题）SiLU 的公式是________；SwiGLU 会把 SiLU 用在________分支上，再乘以 `up`。


2. （填空题）GeGLU 会把 GELU 或 GELU 近似用在________分支上，再乘以 `up`。


3. （填空题）门控激活需要 `gate` 和 `up` 两路输入，其中 `gate` 负责________，`up` 负责________。

4. （选择题）Softmax 中 `row_max` 的 shape 为什么是 `[B, S, H, 1]`？  
   A. 因为沿最后一维规约并使用 `keepdim=True`  
   B. 因为 batch 维被删除  
   C. 因为最后一维必须固定为 1 个 head  
   D. 因为输出 dtype 改成了 INT32


5. （填空题）`pypto.loop` 在 Softmax kernel 中主要用于________。


6. （选择题）如果把 `output_tensor[b_offset:b_offset_end, ...]` 写成过大的切片，可能出现什么问题？  
   A. 覆盖不属于当前 tile 的输出区域  
   B. 自动提升数值精度  
   C. 自动修复动态 shape  
   D. 只影响打印内容

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/04.02_answer.txt


## 10. 小结

到这里，你已经从两个角度练习了算子组合：激活函数展示 elementwise 组合，Softmax 展示 reduction、广播、elementwise 和 loop 的组合。SiLU、GELU、SwiGLU、GeGLU 用来熟悉公式到计算图的翻译方式；Softmax 则进一步引入动态 batch、稳定数值计算和分块写回。

复盘代码时，可以始终抓住四个问题：公式是什么、shape 如何变化、kernel 如何写回输出、PyTorch reference 是否同公式。下一节会继续沿着这条线，把 reduction、广播、matmul 和激活函数组合成 LayerNorm、RMSNorm 和 FFN。